# Download GLORYS data from Copernicus Marine

GLORYS is the CMEMS global ocean physics reanalysis. This notebook uses the
`copernicusmarine` toolbox to subset it by region, time, depth, and variables.

Run in the `data-access-ai4ocean2026` conda env. First time only, log in once:

```bash
copernicusmarine login
```
This generates a file with credentials, by default at `~/.copernicusmarine/.copernicusmarine-credentials`.

If the env doesn't show up for jupyter, you may need to try this
```bash
conda run -n data-access-ai4ocean2026 python -m ipykernel install --user --name data-access-ai4ocean2026
```

Two steps:
1. **Download** — `download_glorys` calls `copernicusmarine.subset` and writes a NetCDF.
2. **Rechunk** — `rechunk_to_zarr` opens with `xr.open_mfdataset`, rechunks, and writes a Zarr store.

In [1]:
from download_glorys import download_glorys, rechunk_to_zarr, DEFAULT_DATASET_ID, DEFAULT_VARIABLES

/Users/aidanjanney/.local/share/mamba/envs/data-access-ai4ocean2026/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Set parameters

In [2]:
region    = dict(min_lon=-80, max_lon=-60, min_lat=30, max_lat=45) # NW Atlantic
time      = dict(start_datetime="2020-01-01", end_datetime="2020-01-31")
variables = DEFAULT_VARIABLES  # temperature, salinity

## 2. Peek at the dataset (metadata only, no download)

In [3]:
import copernicusmarine

ds = copernicusmarine.open_dataset(
    dataset_id=DEFAULT_DATASET_ID,
    variables=variables,
    minimum_longitude=region["min_lon"],
    maximum_longitude=region["max_lon"],
    minimum_latitude=region["min_lat"],
    maximum_latitude=region["max_lat"],
    start_datetime=time["start_datetime"],
    end_datetime=time["end_datetime"],
)
ds

INFO - 2026-07-23T18:15:03Z - Selected dataset version: "202311"
INFO - 2026-07-23T18:15:03Z - Selected dataset part: "default"


<xarray.Dataset> Size: 2GB
Dimensions:    (depth: 50, latitude: 181, longitude: 241, time: 31)
Coordinates:
  * depth      (depth) float32 200B 0.494 1.541 2.646 ... 5.275e+03 5.728e+03
  * latitude   (latitude) float32 724B 30.0 30.08 30.17 ... 44.83 44.92 45.0
  * longitude  (longitude) float32 964B -80.0 -79.92 -79.83 ... -60.08 -60.0
  * time       (time) datetime64[ns] 248B 2020-01-01 2020-01-02 ... 2020-01-31
Data variables:
    thetao     (time, depth, latitude, longitude) float64 541MB dask.array<chunksize=(2, 50, 181, 241), meta=np.ndarray>
    so         (time, depth, latitude, longitude) float64 541MB dask.array<chunksize=(2, 50, 181, 241), meta=np.ndarray>
    uo         (time, depth, latitude, longitude) float64 541MB dask.array<chunksize=(2, 50, 181, 241), meta=np.ndarray>
    vo         (time, depth, latitude, longitude) float64 541MB dask.array<chunksize=(2, 50, 181, 241), meta=np.ndarray>
    zos        (time, latitude, longitude) float64 11MB dask.array<chunksize=(2, 181, 241), meta=np.ndarray>
Attributes: (12/25)
    Conventions:               CF-1.4
    bulletin_date:             2021-07-07 00:00:00
    bulletin_type:             operational
    comment:                   CMEMS product
    domain_name:               GL12
    easting:                   longitude
    ...                        ...
    references:                http://www.mercator-ocean.fr
    source:                    MERCATOR GLORYS12V1
    title:                     daily mean fields from Global Ocean Physics An...
    z_max:                     5727.9169921875
    z_min:                     0.49402499198913574
    copernicusmarine_version:  2.4.1

## 3. Step 1 — Download to NetCDF

In [4]:
nc_path = download_glorys(
    variables=variables,
    **region,
    **time,
    output_filename="glorys_nwatlantic_jan2020.nc",
    output_dir="data",
)
nc_path

INFO - 2026-07-23T18:23:12Z - Selected dataset version: "202311"
INFO - 2026-07-23T18:23:12Z - Selected dataset part: "default"
100%|██████████| [03:57<00:00]  
INFO - 2026-07-23T18:27:19Z - Total size of the download: 645.17 MB.


Downloaded data/glorys_nwatlantic_jan2020.nc


'data/glorys_nwatlantic_jan2020.nc'

## 4. Step 2 — Rechunk to Zarr

In [ ]:
rechunk_to_zarr(
    input_path=nc_path,
    output_path="data/glorys_nwatlantic_jan2020.zarr",
    chunks={"time": 1, "latitude": -1, "longitude": -1},
)

### With dask parallel I/O (useful for large files / many input files)

In [ ]:
from dask.distributed import Client

client = Client()  # open client.dashboard_link to watch progress
client

In [ ]:
rechunk_to_zarr(
    input_path=nc_path,
    output_path="data/glorys_nwatlantic_jan2020.zarr",
    chunks={"time": 1, "latitude": -1, "longitude": -1},
    use_dask=True,
)

client.close()

## 5. Read the Zarr store back

In [ ]:
import xarray as xr

ds_zarr = xr.open_zarr("data/glorys_nwatlantic_jan2020.zarr")
ds_zarr